In [0]:

# Notebook: 02_bronze_layer.py
# Purpose : Parse raw JSON → Delta tables, add metadata columns

from pyspark.sql import SparkSession
from pyspark.sql.functions import col, lit, current_timestamp, to_date
from pyspark.sql.types import StringType
import json
import sys
import pandas as pd
import requests, uuid
from datetime import datetime
from importlib import reload


run_id = str(uuid.uuid4())   

# Add repository root to Python path for module imports
repo_root = "/Workspace/Users/vyshnavi.thunuguntla@outlook.com/databricks-repo-FHIR"
if repo_root not in sys.path:
    sys.path.insert(0, repo_root)

# Import and reload to pick up latest changes
import audit_utils
reload(audit_utils)
from audit_utils import audit_start, audit_end

# Initialize SparkSession BEFORE using it
spark = SparkSession.builder.getOrCreate()

from config.config import RAW_PATH, BRONZE_PATH, BRONZE_DB, RESOURCES, END_DATE
from utils.fhir_utils import extract_entries, add_metadata

spark.sql(f"CREATE DATABASE IF NOT EXISTS {BRONZE_DB}")


def load_bronze(resource: str):
    raw_file = f"{RAW_PATH}/{resource}/date={END_DATE}/bundle.json"

    with open(raw_file) as f:
        pages = json.load(f)

    all_records = []
    api_url = f"https://hapi.fhir.org/baseR4/{resource}"

    for bundle in pages:
        entries = extract_entries(bundle)
        entries = add_metadata(entries, api_url, resource)
        all_records.extend(entries)

    if not all_records:
        print(f"[BRONZE] {resource}: No records found.")
        return

    # Convert to Spark DataFrame via pandas (serverless-compatible)
    # Create pandas DataFrame from list of dicts, then convert to Spark
    pandas_df = pd.DataFrame(all_records)
    df = spark.createDataFrame(pandas_df)

    # Ensure metadata columns exist
    for col_name in ["_extraction_timestamp", "_api_url", "_resource_type", "_ingestion_date"]:
        if col_name not in df.columns:
            df = df.withColumn(col_name, lit(None).cast(StringType()))

    df = df.withColumn("load_timestamp", current_timestamp()) \
           .withColumn("partition_date", to_date(col("_ingestion_date")))

    table_name = f"{BRONZE_DB}.{resource.lower()}"
    (
        df.write
          .format("delta")
          .mode("append")
          .partitionBy("partition_date")
          .option("mergeSchema", "true")
          .saveAsTable(table_name)
    )
    print(f"[BRONZE] {resource}: {df.count()} rows → {table_name}")

for res in RESOURCES:
    table_name = f"{BRONZE_DB}.{res.lower()}"
    CNT1 = spark.sql(f"SELECT COUNT(*) FROM {table_name} ").collect()[0][0]
    started_at = datetime.utcnow() # current_timestamp(),
    audit_start(
        run_id        = run_id,
        stage         = "BRONZE",
        resource_type = res,
        operation     = "INGEST",
        status        = "STARTED",
        started_at = started_at ,
        before_run_source_count = CNT1 ,
        created_by    = "01_bronze_ingest"
    )
    load_bronze(res)
    CNT2 = spark.sql(f"SELECT COUNT(*) FROM {table_name} ").collect()[0][0]
    audit_end(
        run_id        = run_id,
        stage         = "BRONZE",
        resource_type = res,
        operation     = "INGEST",
        status        = "ENDED",
        post_run_source_count = CNT2 ,
    )

print("✅ Bronze layer complete.")